# Lab C — Desplegar tu modelo custom (fusionado) como un Vertex AI Endpoint

Los Labs A y B de `class04/` cubrían dos extremos: en el **Lab A** desplegaron
un modelo *ya empaquetado* del catálogo de Model Garden con un solo
`.deploy()`; en el **Lab B** montaron ustedes mismos `vLLM` sobre una VM sin
pasar por ningún "endpoint" formal de GCP. Este Lab C cubre el punto
intermedio: tomar el modelo que **ustedes entrenaron** (el Gemma-7b-it+LoRA
de la sección de fine-tuning, ya sea entrenado en la VM+JupyterHub o con un
Custom Training Job de Vertex AI) y ponerlo en el **Vertex AI Model
Registry**, servido por un **Endpoint** gestionado (autoscaling, IAM,
`predict()` con SDK/REST) -la forma "seria" de exponer un modelo custom.

## Dos variantes para servir un modelo LoRA -esta es la variante A

Existen dos formas de llevar un modelo afinado con LoRA a un endpoint de
Vertex AI:

- **Fusionado (este notebook, Lab C):** se fusionan los adaptadores LoRA
  dentro de los pesos del modelo base (`merge_and_unload()`), se sube el
  modelo ya fusionado a Cloud Storage, y se despliega como un modelo normal.
  Es el camino más simple y compatible.
- **LoRA dinámico (Lab D, notebook aparte):** se despliega el modelo *base*
  sin tocar, y el adaptador se pasa en cada petición de predicción -sin
  fusionar nada, y permitiendo servir varios adaptadores desde un solo
  endpoint. Ver `class04d-deploy-dynamic-lora.ipynb`.

## Arquitectura de este lab

```
gs://bucket/gemma-7b-it-samsum-lora/   (adaptador LoRA, del Custom Training Job)
        │
        ▼
  fusionar con el modelo base (esta VM+JupyterHub, con GPU)
        │
        ▼
gs://bucket/gemma-7b-it-samsum-merged/  (modelo completo, fusionado)
        │
        ▼
  aiplatform.Model.upload(...)   -> Vertex AI Model Registry
        │
        ▼
  aiplatform.Endpoint.create() + model.deploy()   -> Endpoint con GPU
        │
        ▼
  endpoint.predict(...)   -> resúmenes generados
```

**Requisito:** haber terminado el fine-tuning de `train_gemma.py`/
`train_gemma_vertex` (Custom Training Job) y tener la ruta `gs://.../gemma-7b-it-samsum-lora`
de los adaptadores.

Corre este notebook en el mismo contenedor JupyterHub de los labs de
fine-tuning (tiene GPU y ya trae `torch`/`transformers`/`peft` instalados) -la
fusión de pesos necesita GPU; el despliegue y las predicciones, una vez
subido el modelo a GCS, ya no.


## 0. Instalar dependencias

In [ ]:
%pip install torch transformers peft accelerate huggingface_hub google-cloud-storage google-cloud-aiplatform


## 1. Configuración

Cambia `PROJECT_ID`, `GCS_ADAPTER_DIR` (la salida del entrenamiento) y
`GCS_MERGED_DIR` (dónde vas a dejar el modelo fusionado -mismo bucket, otra
carpeta).


In [ ]:
import os

PROJECT_ID = "myproyectsi4002-262"          # <-- reemplaza esto
LOCATION = "us-central1"

BASE_MODEL_NAME = "google/gemma-7b-it"

GCS_ADAPTER_DIR = "gs://si7016emontoya2/gemma-7b-it-samsum-lora"     # <-- salida del entrenamiento
GCS_MERGED_DIR = "gs://si7016emontoya2/gemma-7b-it-samsum-merged"    # <-- donde se sube el modelo fusionado

LOCAL_ADAPTER_DIR = "/home/jovyan/labs/gemma-7b-it-samsum-lora-vertex"
LOCAL_MERGED_DIR = "/home/jovyan/labs/gemma-7b-it-samsum-merged"

MACHINE_TYPE = "g2-standard-4"
ACCELERATOR_TYPE = "NVIDIA_L4"
ACCELERATOR_COUNT = 1

# Opcional: cuenta de servicio con permiso de lectura sobre el bucket, si el
# endpoint necesita credenciales distintas a las de la cuenta de servicio por
# defecto del proyecto.
SERVICE_ACCOUNT = None

for v in (PROJECT_ID, GCS_ADAPTER_DIR, GCS_MERGED_DIR):
    assert not v.startswith("["), f"Falta reemplazar un placeholder: {v!r}"


## 2. Descargar el adaptador LoRA desde Cloud Storage

Mismo helper que en `infer_gemma_vertex.py`/`evaluate_gemma_vertex.py`: usa el
cliente de Python `google-cloud-storage` (no `gcloud`/`gsutil`, que no viene
instalado en esta imagen) con credenciales automáticas del servidor de
metadata de la VM.


In [ ]:
def download_gcs_dir(gcs_uri, local_dir, force=False):
    if os.path.isdir(local_dir) and os.listdir(local_dir) and not force:
        print(f"Ya existe una copia local en {local_dir}. Saltando descarga.")
        return local_dir

    from google.cloud import storage

    bucket_name, _, prefix = gcs_uri[len("gs://"):].partition("/")
    prefix = prefix.rstrip("/")

    print(f"Descargando {gcs_uri} -> {local_dir} ...")
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs = [b for b in bucket.list_blobs(prefix=prefix + "/" if prefix else prefix)
             if not b.name.endswith("/")]

    if not blobs:
        raise SystemExit(f"No encontré archivos en {gcs_uri}. ¿Es correcta la ruta?")

    os.makedirs(local_dir, exist_ok=True)
    for blob in blobs:
        rel_path = blob.name[len(prefix):].lstrip("/") if prefix else blob.name
        dest = os.path.join(local_dir, rel_path)
        os.makedirs(os.path.dirname(dest) or local_dir, exist_ok=True)
        blob.download_to_filename(dest)

    print(f"Descarga completa: {len(blobs)} archivo(s) en {local_dir}")
    return local_dir


def upload_gcs_dir(local_dir, gcs_uri):
    """Lo inverso de download_gcs_dir: sube todo el contenido de local_dir a
    gcs_uri (una carpeta gs://...)."""
    from google.cloud import storage

    bucket_name, _, prefix = gcs_uri[len("gs://"):].partition("/")
    prefix = prefix.rstrip("/")

    client = storage.Client()
    bucket = client.bucket(bucket_name)

    archivos = []
    for root, _, files in os.walk(local_dir):
        for fname in files:
            archivos.append(os.path.join(root, fname))

    print(f"Subiendo {len(archivos)} archivo(s) {local_dir} -> {gcs_uri} ...")
    for path in archivos:
        rel_path = os.path.relpath(path, local_dir)
        blob_name = f"{prefix}/{rel_path}" if prefix else rel_path
        bucket.blob(blob_name).upload_from_filename(path)
        print(f"  subido: {rel_path}")

    print(f"Subida completa a {gcs_uri}")


ADAPTER_DIR = download_gcs_dir(GCS_ADAPTER_DIR, LOCAL_ADAPTER_DIR)
print("Adaptadores disponibles en:", ADAPTER_DIR)


## 3. Login en Hugging Face

Necesario para descargar los pesos del modelo base (`google/gemma-7b-it` es
*gated*) y fusionarlos con el adaptador.


In [ ]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
login(token=hf_token)
del hf_token


## 4. Cargar el modelo base en bfloat16 (NO en 4-bit) y fusionar el LoRA

**Por qué bf16 y no 4-bit:** `merge_and_unload()` suma los deltas de LoRA
directamente a los pesos del modelo. Esa suma requiere pesos en precisión
completa (o bf16/fp16) -sobre un modelo cuantizado a 4-bit (como el que
usamos para *entrenar*, con `BitsAndBytesConfig`) el resultado de la fusión
no es correcto. Por eso aquí cargamos el modelo base "normal", sin
cuantizar, solo para este paso.

**Nota de memoria:** Gemma-7b-it en bf16 ocupa ~14GB de VRAM -cabe en la GPU
L4 de 24GB de esta VM, pero sin mucho margen. Si te quedas sin memoria,
reinicia el kernel para liberar la VRAM que quedó del fine-tuning/inferencia
anteriores antes de correr esta celda.


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print("CUDA disponible:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

modelo_con_lora = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
print("Fusionando adaptadores LoRA en los pesos del modelo base...")
modelo_fusionado = modelo_con_lora.merge_and_unload()
print("Fusión completa.")


## 5. Guardar el modelo fusionado localmente

In [ ]:
os.makedirs(LOCAL_MERGED_DIR, exist_ok=True)
modelo_fusionado.save_pretrained(LOCAL_MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(LOCAL_MERGED_DIR)
print("Modelo fusionado guardado en:", LOCAL_MERGED_DIR)
print(os.listdir(LOCAL_MERGED_DIR))


## 6. Subir el modelo fusionado a Cloud Storage

Este paso puede tardar varios minutos -Gemma-7b-it en bf16 son ~14GB de
archivos `.safetensors`.


In [ ]:
upload_gcs_dir(LOCAL_MERGED_DIR, GCS_MERGED_DIR)


## 7. Liberar la GPU local

A partir de aquí ya no necesitamos el modelo cargado en esta VM -todo lo que
sigue pasa por la API de Vertex AI contra el modelo que acabamos de subir a
GCS.


In [ ]:
del modelo_fusionado, modelo_con_lora, base_model
torch.cuda.empty_cache()
print("Memoria GPU liberada.")


## 8. Inicializar el SDK de Vertex AI


In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI inicializado: proyecto={PROJECT_ID}, región={LOCATION}")


## 9. El contenedor vLLM prebuilt de Vertex AI

Google publica un contenedor con `vLLM` ya empaquetado para servir modelos
abiertos -no hace falta escribir el servidor de inferencia a mano, solo
pasarle los argumentos de vLLM (igual que en la línea de comandos del Lab B,
pero en formato `serving_container_args`).

**Revisa la versión más reciente** en la documentación de Model Garden antes
de usar esto en producción -las imágenes se actualizan periódicamente.


In [ ]:
VLLM_DOCKER_URI = (
    "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/"
    "pytorch-vllm-serve:20241210_0916_RC00"
)


## 10. Función de despliegue (Model Registry -> Endpoint)

Adaptada del patrón oficial que usa Vertex AI Model Garden para desplegar
modelos con vLLM (`aiplatform.Model.upload` + `aiplatform.Endpoint.create` +
`model.deploy`). El endpoint expone la ruta `/generate` del servidor de vLLM
(`vllm.entrypoints.api_server`), que espera instancias con `prompt`,
`max_tokens`, `temperature`, etc.


In [ ]:
from typing import Optional, Tuple


def deploy_model_vllm(
    model_name: str,
    model_id: str,
    machine_type: str = MACHINE_TYPE,
    accelerator_type: str = ACCELERATOR_TYPE,
    accelerator_count: int = ACCELERATOR_COUNT,
    gpu_memory_utilization: float = 0.85,
    max_model_len: int = 2048,
    dtype: str = "bfloat16",
    enable_lora: bool = False,
    max_loras: int = 1,
    max_cpu_loras: int = 4,
    env_vars: Optional[dict] = None,
    service_account: Optional[str] = None,
) -> Tuple["aiplatform.Model", "aiplatform.Endpoint"]:
    vllm_args = [
        "python", "-m", "vllm.entrypoints.api_server",
        "--host=0.0.0.0", "--port=8080",
        f"--model={model_id}",
        f"--tensor-parallel-size={accelerator_count}",
        "--swap-space=16",
        f"--gpu-memory-utilization={gpu_memory_utilization}",
        f"--max-model-len={max_model_len}",
        f"--dtype={dtype}",
        f"--max-loras={max_loras}",
        f"--max-cpu-loras={max_cpu_loras}",
        "--disable-log-stats",
    ]
    if enable_lora:
        vllm_args.append("--enable-lora")

    model = aiplatform.Model.upload(
        display_name=model_name,
        serving_container_image_uri=VLLM_DOCKER_URI,
        serving_container_args=vllm_args,
        serving_container_ports=[8080],
        serving_container_predict_route="/generate",
        serving_container_health_route="/ping",
        serving_container_environment_variables=env_vars or {},
        serving_container_shared_memory_size_mb=16 * 1024,
        serving_container_deployment_timeout=7200,
    )
    print("Modelo registrado:", model.resource_name)

    endpoint = aiplatform.Endpoint.create(display_name=f"{model_name}-endpoint")
    print("Endpoint creado:", endpoint.resource_name)

    print("Desplegando (esto puede tardar 10-20 minutos: descarga del modelo + arranque de vLLM)...")
    model.deploy(
        endpoint=endpoint,
        machine_type=machine_type,
        accelerator_type=accelerator_type,
        accelerator_count=accelerator_count,
        deploy_request_timeout=1800,
        service_account=service_account,
    )
    print("Despliegue completo.")
    return model, endpoint


## 11. Desplegar el modelo fusionado

`model_id` es la ruta de GCS del modelo fusionado -vLLM la lee directamente
desde Cloud Storage (soporta descarga paralela desde `gs://`), no hace falta
montarla como disco.


In [ ]:
modelo_vertex, endpoint = deploy_model_vllm(
    model_name="gemma-7b-it-samsum-lora-merged",
    model_id=GCS_MERGED_DIR,
    enable_lora=False,   # ya está fusionado, no se necesita LoRA en tiempo de inferencia
    service_account=SERVICE_ACCOUNT,
)


## 12. Probar el endpoint

Construimos el prompt con el mismo `apply_chat_template` que usamos en
`infer_gemma.py`/`infer_gemma_vertex.py` -el tokenizer ya lo tenemos cargado
(paso 4) del propio directorio del adaptador.


In [ ]:
def build_prompt(dialogue):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogue}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resumir_endpoint(dialogue, max_tokens=64, temperature=0.0):
    prompt = build_prompt(dialogue)
    instance = {
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": 1.0,
        "top_k": -1,
    }
    response = endpoint.predict(instances=[instance])
    return response.predictions[0]


In [ ]:
ejemplos = [
    "Carlos: ¿Vas a venir a la reunión de las 3pm?\n"
    "Marta: Sí, ya salgo. ¿La sala sigue siendo la 402?\n"
    "Carlos: Sí, misma sala. Nos vemos ahí.",

    "Ana: hey are we still on for the gym at 6?\n"
    "Leo: yeah but running a bit late, more like 6:20\n"
    "Ana: no worries, I'll grab a locker and wait",
]

for i, dialogo in enumerate(ejemplos, start=1):
    print(f"\n{'=' * 70}\nEjemplo {i}\n{'=' * 70}")
    print("Diálogo:\n" + dialogo)
    print("\n>> Resumen (endpoint, modelo fusionado):\n" + resumir_endpoint(dialogo))


## 13. Costos y buenas prácticas

- La GPU del endpoint se cobra **por hora mientras el modelo esté
  desplegado** en el endpoint -igual que en el Lab A, no hay "escalar a
  cero" automático salvo que configures `min_replica_count=0` (no todas las
  combinaciones de máquina/acelerador lo permiten).
- El modelo fusionado en GCS (~14GB en bf16) también genera costo de
  almacenamiento -bórralo si no lo vas a volver a desplegar (`gsutil -m rm -r`
  o desde la consola).
- Si vas a servir varios adaptadores distintos entrenados sobre el mismo
  modelo base, fusionar cada uno por separado significa un modelo completo
  (~14GB) por adaptador en GCS y, peor, un endpoint por modelo si quieres
  tenerlos disponibles a la vez -ver el Lab D (LoRA dinámico) para el caso en
  que eso importa.


## 14. Limpieza obligatoria

In [ ]:
endpoint.undeploy_all()
endpoint.delete()
modelo_vertex.delete()
print("Endpoint y modelo eliminados. Ya no se cobra por este despliegue.")
print("Recuerda borrar también", GCS_MERGED_DIR, "si no lo vas a reutilizar (costo de almacenamiento).")


## Ejercicio

1. Despliega el mismo modelo fusionado con `dtype="float16"` en vez de
   `"bfloat16"` y compara si cambia la latencia o la calidad de los
   resúmenes.
2. Sube `max_model_len` y mide si el tiempo de arranque del endpoint
   (`model.deploy(...)`) cambia de forma notoria.
3. Compara la latencia de `endpoint.predict(...)` de este Lab C contra la del
   Lab B (llamando directamente al vLLM de la VM por el túnel SSH) para el
   mismo diálogo -¿cuál responde más rápido y por qué podría ser?
4. Estima el costo mensual de dejar este endpoint desplegado 24/7 con la
   calculadora de precios de GCP, y compáralo con el costo de una VM del Lab
   B que se apaga entre clases.
5. Lee `class04d-deploy-dynamic-lora.ipynb` y, sin ejecutarlo todavía,
   describe en dos frases cuándo elegirías LoRA dinámico sobre fusionar -pista:
   piensa en cuántos adaptadores distintos vas a servir.

## Notas finales

- El equivalente sin notebook (usando solo `gcloud`, asumiendo que el modelo
  fusionado ya está en GCS) está en `class04c-deploy-gcloud.sh`.
- Este mismo patrón (`aiplatform.Model.upload` + contenedor vLLM prebuilt) es
  el que usa Model Garden internamente cuando en el Lab A llamaste a
  `OpenModel(...).deploy(...)` -aquí lo controlas manualmente porque el
  modelo no viene del catálogo, es el tuyo.
- Ver `class04d-deploy-dynamic-lora.ipynb` para la variante que no fusiona
  nada y sirve el adaptador directamente desde GCS en cada petición.
